In [0]:
env = "dev"
silverTablName = f"saleslake_{env}.silver_schema_{env}.cleanedinvoice"
goldTablName = f"saleslake_{env}.gold_schema_{env}.fact_invoice"

# Note: fact_invoice does not support SCD2 - it's a fact table, not a dimension
# This simplified version loads new invoices only (no updates)
spark.sql(f"""
INSERT INTO {goldTablName}
SELECT DISTINCT
    s.invoice_id,
    s.invoice_number,
    NULL as customer_sk,
    NULL as store_sk,
    NULL as region_sk,
    NULL as discount_sk,
    NULL as invoice_date_sk,
    NULL as due_date_sk,
    NULL as payment_date_sk,
    s.subtotal_amount,
    s.discount_amount,
    s.tax_amount,
    s.total_amount,
    DATEDIFF(s.payment_date, s.invoice_date) as days_to_pay,
    s.payment_status,
    CASE WHEN s.payment_status = 'OVERDUE' OR (s.payment_status = 'PENDING' AND CURRENT_DATE() > s.due_date) THEN 'Y' ELSE 'N' END as is_overdue,
    s.invoice_date,
    CURRENT_TIMESTAMP() as load_ts
FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY invoice_id ORDER BY ingest_ts DESC) as rn
    FROM {silverTablName}
    WHERE ingest_ts > (SELECT COALESCE(MAX(load_ts), TO_TIMESTAMP('1990-01-01','yyyy-MM-dd')) FROM {goldTablName})
) s
WHERE s.rn = 1
  AND NOT EXISTS (SELECT 1 FROM {goldTablName} g WHERE g.invoice_id = s.invoice_id)
""")

In [0]:
%sql
SELECT * FROM saleslake_dev.gold_schema_dev.fact_invoice;
SELECT * FROM saleslake_dev.silver_schema_dev.cleanedinvoice;

In [0]:
%sql
describe extended saleslake_dev.gold_schema_dev.refinedinvoice;
describe extended saleslake_dev.silver_schema_dev.cleanedinvoice;

In [0]:
%sql
describe extended saleslake_dev.gold_schema_dev.refinedinvoice;